# From survey responses to a reproducible submission

This notebook explains our NumPy-only GBM–MLP pipeline and provides a path from an experiment to a defensible report. It starts with the prediction problem, follows the data through the models, and ends with model selection and submission. Expensive training is **opt-in**; reading or running the notebook does not automatically train the ensemble.

The supplied leaderboard screenshot reports F1 **0.455** and accuracy **0.883**. These are user-reported public leaderboard results, not measurements made by this notebook. A local improvement does not guarantee a public or private leaderboard improvement.

## Notebook controls — no terminal required
Run the setup cell, then edit its switches. For a submission, set `RUN_TRAINING=True`
and run the training cell in section 7. For local evaluation, set
`RUN_VALIDATION=True` and run section 6. Progress streams directly into the notebook.
The default switches are off to prevent an accidental long job with Run All.
After one fit, rerun only the blend cell to try another weight/rate. Use distinct
output directories when changing training settings, and do not run two heavy fits
at once. The shell examples in the explanation are optional equivalents.


In [1]:
from pathlib import Path
import os, sys
import numpy as np
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "run.py").exists() and (p / "preprocessing.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the extracted project folder.")
sys.path.insert(0, str(ROOT))
# Legacy notebooks use ../ paths, so give every notebook the same working directory.
os.chdir(ROOT / "notebooks")
print("Project:", ROOT)
import json
from metrics import precision_recall_f1, predict_labels
import run as submission_pipeline
import cv_final as validation_pipeline
import reblend as blending_pipeline

# Edit these settings once; all following cells reuse them.
DATA_PATH = ROOT / "data/dataset"
OUTPUT_DIR = ROOT / "submissions/notebook_run"
CV_DIR = ROOT / "experiments/results/notebook_cv"
WITH_LEAKAGE = True
MLP_SPLIT_PER_SEED = True
GBM_WEIGHT = 0.50
TOP_RATE = 0.128 if WITH_LEAKAGE else 0.136
RUN_VALIDATION = False   # True: run the validation cell below
RUN_TRAINING = False     # True: run the submission-training cell below
CV_FOLDS = 0            # 0: one holdout fit; 5: development CV + holdout

common = ["--data_path", str(DATA_PATH), "--gbm_weight", str(GBM_WEIGHT),
          "--top_rate", str(TOP_RATE)]
if WITH_LEAKAGE:
    common.append("--with_leakage")
if MLP_SPLIT_PER_SEED:
    common.append("--mlp_split_per_seed")


Project: /workspace/scratch/7a31394ff660/ML_Project1-zhihao


## 1. Define the prediction problem before choosing a model

Each row contains BRFSS survey responses. Training labels encode heart disease as +1 and the negative class as −1. The test labels are unknown. We must preserve the test IDs and their order when writing `Id,Prediction`.

The classes are imbalanced, so accuracy alone can hide poor detection of positives. Our main selection criterion is positive-class F1:

$$P=TP/(TP+FP),\quad R=TP/(TP+FN),\quad F_1=2TP/(2TP+FP+FN).$$

Increasing the number of predicted positives generally trades precision for recall. This makes the operating point part of model selection, not just a formatting choice.

In [2]:
# Inspect labels without loading the large feature matrices.
label_path = DATA_PATH / "y_train.csv"
if label_path.exists():
    labels = np.loadtxt(label_path, delimiter=",", skiprows=1)[:, -1]
    print("Rows:", len(labels), "positive share:", np.mean(labels == 1))
else:
    print("Place the supplied dataset in data/dataset/ to inspect it.")


Rows: 328135 positive share: 0.08830207079403295


## 2. Separate the competition track from the clinical question

`model_inputs` in `run.py` calls `clean_data`, retains `_STATE` for the tree pipeline, and adds composite health features. Survey special codes need variable-specific interpretation: a code meaning 'none' is different from 'refused' or 'unknown'. Consult `preprocessing.py` for the actual mappings rather than applying one global rule.

With `--with_leakage`, `leaky_features` adds diagnosis-dependent survey questions. The existing pipeline also boosts scores by 10 when the cleaned HAREHAB1 answer is present. This is a survey skip-pattern heuristic, **not a guarantee about unseen test labels**. It can exploit information unavailable before diagnosis. Describe this track transparently in the report and evaluate a clean track separately; competition usefulness does not establish clinical validity.

Missing-value handling differs by learner. The tree path handles binned features and missingness; the neural path fits preprocessing on the outer training rows, transforms both partitions, and adds standardized leakage flags when enabled.

## 3. Split first, then fit transformations

For each outer validation fold, `cv_final.py` passes only the outer training labels to `ensemble_scores`. Risk encodings, scaling and models are fitted there, then applied to the held-out rows. The six categorical risk encodings use a smoothed prevalence:

$$e(c)=rac{n_{+,c}+50\,\bar y}{n_c+50}.$$

Unseen categories fall back to the training prior. `_STATE` has an additional encoding in the tree model. The six risk encodings are in-sample on outer training rows and are computed before each learner's inner early-stopping split. Consequently, inner validation loss is not a fully independent estimate; **outer held-out F1 is the relevant evidence**. Cross-fitting target encodings is a future controlled experiment, not an implemented claim here.

Use development CV to choose settings, then evaluate a locked candidate on an untouched holdout. Repeatedly tuning on the same holdout turns it into development data. The two earlier screening partitions overlap and must not be described as independent folds.

## 4. Learn complementary scores

The tree ensemble averages three GBMs (seeds 42, 43, 44, shifted by fold). Defaults allow up to 1,500 rounds with early stopping. Trees capture thresholds and interactions without requiring the same scaling as a neural network.

The neural ensemble averages logits from ten networks, each with one 64-unit ReLU hidden layer. It uses minibatch Adam, binary cross-entropy, weight decay and early stopping. `--mlp_split_per_seed` varies the inner split so every network does not omit exactly the same validation rows. This option changes training coverage; it is **not a runtime shortcut**.

`--gbm_refit_full` selects rounds on an inner split and then fits again using all outer training rows. It costs extra time, and earlier screening did not establish a consistent gain when combined with varied MLP splits. Leave it off unless a separate experiment supports it.

The GBM and MLP return scores on a logit-like scale. We average their raw scores, not calibrated probabilities:

$$s=w\,s_{GBM}+(1-w)\,s_{MLP}+b_{HAREHAB1}.$$

The default is $w=0.50$. The optional candidate uses $w=0.55$. Adding the leakage boost **after** blending prevents its weight from changing with the mixture.

## 5. Choose the operating point and examine the evidence

The pipeline predicts +1 for scores at or above the $(1-r)$ quantile. The default leakage-track rate is $r=0.128$; a candidate is $0.129$. This controls an approximate positive fraction, not a probability cutoff. Ties may make the selected fraction slightly larger. Applying a fixed quantile rule to an unlabeled test batch does not use its labels, but it assumes a stable operating fraction across batches.

Historical project notes suggested a 55/45 blend. Re-evaluating the existing reduced-budget screening scores does not confirm a general improvement. Those experiments used 60,000 sampled rows, 45,000 training rows per split, one GBM (at most 300 trees), and three MLP seeds. These are **not the full production ensemble**. The table below compares settings on identical cached predictions; no new models were trained for this comparison.

In [3]:
results = json.loads((ROOT / "experiments/results/blend_screening.json").read_text())
print("weight rate     split suffix       F1       precision recall")
for row in results:
    print(f"{row['gbm_weight']:.2f}   {row['top_rate']:.3f}    {row['split'][-15:]:15} "
          f"{row['f1']:.6f} {row['precision']:.6f} {row['recall']:.6f}")
for w, r in [(0.50, 0.128), (0.55, 0.129)]:
    values = [x['f1'] for x in results if x['gbm_weight'] == w and x['top_rate'] == r]
    print(f"Mean screening F1 ({w=}, {r=}): {np.mean(values):.6f}")


weight rate     split suffix       F1       precision recall
0.50   0.128    20261027_scores 0.459368 0.390104 0.558538
0.55   0.128    20261027_scores 0.457528 0.388542 0.556301
0.50   0.129    20261027_scores 0.459096 0.388630 0.560776
0.55   0.129    20261027_scores 0.457265 0.387080 0.558538
0.50   0.128    20261028_scores 0.441582 0.375000 0.536913
0.55   0.128    20261028_scores 0.443422 0.376563 0.539150
0.50   0.129    20261028_scores 0.441392 0.373643 0.539150
0.55   0.129    20261028_scores 0.442002 0.374160 0.539896
Mean screening F1 (w=0.5, r=0.128): 0.450475
Mean screening F1 (w=0.55, r=0.129): 0.449634


The default averages approximately **0.450475**, versus **0.449634** for the candidate. One split improves and the other worsens. Therefore we retain the default and make the candidate selectable. Do not write that the new blend improves leaderboard F1: that has not been demonstrated.

The practical improvement in this release is experiment efficiency: component scores are retained, so model fitting is paid for once and blend/rate comparisons are cheap.

## 6. Validate once, compare cached scores cheaply

The command below performs the existing five-fold development CV on 80% of the data and evaluates the selected configuration on the remaining 20%. It is expensive (six ensemble fits). Choose settings using only `fold_*.npz`; keep `holdout.npz` out of tuning. If you change settings after inspecting the holdout, disclose that it is no longer untouched.

For a single predeclared candidate evaluation use `--holdout --k_fold 0`, which performs one fit. This is faster but does not provide development CV for choosing settings. Use distinct score directories for different training configurations.

In [4]:
# Calls Python directly in this kernel: progress appears below the cell.
# Kernel Interrupt stops training. No terminal or subprocess is used.
if RUN_VALIDATION:
    validation_pipeline.main(common + ["--holdout", "--k_fold", str(CV_FOLDS),
                                       "--scores_dir", str(CV_DIR)])
else:
    print("Validation skipped. Set RUN_VALIDATION=True above to enable it.")


Validation skipped. Set RUN_VALIDATION=True above to enable it.


In [5]:
# Small predeclared comparison on development folds only; no retraining.
paths = sorted(CV_DIR.glob("fold_*.npz"))
for w, rate in [(0.50, 0.128), (0.55, 0.128), (0.50, 0.129), (0.55, 0.129)]:
    fold_f1 = []
    for path in paths:
        with np.load(path, allow_pickle=False) as d:
            scores = w*d['gbm'] + (1-w)*d['mlp'] + d['boost']
            pred = predict_labels(scores, np.quantile(scores, 1-rate))
            fold_f1.append(precision_recall_f1(d['y'], pred)[2])
    if fold_f1:
        print(w, rate, "fold F1:", fold_f1, "mean:", np.mean(fold_f1))
if not paths:
    print("No full-ensemble CV components yet. Screening evidence above is still available.")


No full-ensemble CV components yet. Screening evidence above is still available.


## 7. Train the submission ensemble, then reuse its scores

Run from the project folder:

```bash
python run.py --with_leakage --mlp_split_per_seed --output submissions/baseline.csv
```

This writes the submission `baseline.csv` and the reusable `baseline_components.npz` containing aligned test IDs, GBM scores, MLP scores and the boost. It also retains the legacy blended `.npy` output. You can then generate the alternative without training again:

```bash
python reblend.py --components submissions/baseline_components.npz --gbm_weight 0.55 --top_rate 0.129 --output submissions/candidate_55_129.csv
```

Upload the chosen **CSV** to AIcrowd. `cv_final.py` evaluates models locally; it is not required after generating a submission and does not upload anything. Old combined `.npy` scores cannot be separated into GBM and MLP scores. To change their blend, a single run of the updated training pipeline is needed. To change only their positive rate, the existing `generate_submission_from_scores.py` remains available.

Training progress now includes GBM start/completion and MLP epoch validation loss. The changes do not promise faster model training; they avoid repeating that training during blend selection. Early-stopping losses and elapsed times vary by machine. Full model weights are not checkpointed, so these score files do not resume an interrupted fit.

In [6]:
# Train once; the CSV and component archive are saved together.
BASELINE_CSV = OUTPUT_DIR / "baseline.csv"
COMPONENTS = OUTPUT_DIR / "baseline_components.npz"
if RUN_TRAINING:
    submission_pipeline.main(common + ["--output", str(BASELINE_CSV)])
else:
    print("Training skipped. Set RUN_TRAINING=True above, then run this cell.")
    print("Existing components:", COMPONENTS.exists())


Training skipped. Set RUN_TRAINING=True above, then run this cell.
Existing components: False


In [7]:
# Fast: reuse saved scores; change these two settings without retraining.
CANDIDATE_WEIGHT = 0.55
CANDIDATE_RATE = 0.129
candidate_csv = OUTPUT_DIR / "candidate_55_129.csv"
if COMPONENTS.exists():
    blending_pipeline.main(["--components", str(COMPONENTS),
                           "--gbm_weight", str(CANDIDATE_WEIGHT),
                           "--top_rate", str(CANDIDATE_RATE),
                           "--output", str(candidate_csv)])
    try:
        from IPython.display import FileLink, display
        display(FileLink(os.path.relpath(candidate_csv, Path.cwd())))
    except ImportError:
        print("Submission saved:", candidate_csv)
else:
    print("Train once in the preceding cell, or set COMPONENTS to an existing archive.")


Train once in the preceding cell, or set COMPONENTS to an existing archive.


In [8]:
# Check actual test-ID order and CSV format, using the supplied test header/IDs.
import csv
if candidate_csv.exists():
    with candidate_csv.open() as f:
        reader = csv.reader(f)
        assert next(reader) == ["Id", "Prediction"]
        rows = np.asarray(list(reader), dtype=np.int64)
    with (DATA_PATH / "x_test.csv").open() as f:
        reader = csv.reader(f); next(reader)
        expected_ids = np.fromiter((int(row[0]) for row in reader), dtype=np.int64)
    assert np.array_equal(rows[:, 0], expected_ids), "Test IDs/order mismatch"
    assert np.isin(rows[:, 1], [-1, 1]).all(), "Invalid prediction labels"
    print("Ready for upload:", candidate_csv, "rows:", len(rows))
else:
    print("Generate a CSV above before checking it.")


Generate a CSV above before checking it.


## 8. Turn the workflow into a concise report

Build the report around a question and evidence, rather than a list of scripts:

1. **Problem and protocol:** define the positive label, imbalance, F1 and the development/holdout separation. State which data and tracks are used.
2. **Representation and methods:** explain missing codes, composite features, smoothed category risks, the GBM and the MLP. Give the blend equation and the quantile decision rule.
3. **Controlled comparisons:** compare baseline against varied MLP splits, blend weight, and operating rate with all other settings held fixed. Report precision, recall and F1, training budget, seeds and partition sizes. Distinguish screening, full CV, holdout and leaderboard scores.
4. **Interpretation and limits:** explain whether gains are consistent, acknowledge tuning uncertainty, discuss diagnosis-dependent features, and avoid interpreting leaderboard performance as clinical validity.

Suggested results table columns: configuration; training budget; split; precision; recall; F1; runtime; evidence source. Fill missing full-run or leaderboard results only after measuring them. The JSON in `experiments/results/blend_screening.json` supports the small-budget comparison above; historical notes alone do not establish a new gain.

A defensible conclusion today is: *Caching aligned learner scores enables reproducible blend and threshold comparisons without retraining. The proposed 55/45, 12.9% alternative showed mixed results in reduced-budget screening, so the established default was retained pending full-ensemble validation.*